In [61]:
!git clone https://github.com/crackerghost/reatime-voice-api.git
!git pull origin master

fatal: destination path 'reatime-voice-api' already exists and is not an empty directory.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 595 bytes | 148.00 KiB/s, done.
From https://github.com/crackerghost/reatime-voice-api
 * branch            master     -> FETCH_HEAD
   005c7c4..35160d2  master     -> origin/master
Updating 005c7c4..35160d2
Fast-forward
 voice_api.py | 5 +++--
 1 file changed, 3 insertions(+), 2 deletions(-)


In [10]:
%cd /kaggle/working/reatime-voice-api

/kaggle/working/reatime-voice-api


In [56]:
!python -m pip install -q -r requirements.txt ctranslate2 faster-whisper


In [67]:
import os
from kaggle_secrets import UserSecretsClient

# 1. API Keys
os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

# 2. CUDA Memory & Stability (Prevents fragmentation without empty_cache lag)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# 3. LLM (CRITICAL FOR LATENCY: Replaces slow reasoning model with ultra-fast Llama 3.3)
os.environ["LLM_MODEL"] = "openai/gpt-oss-20b"
os.environ["LLM_TEMPERATURE"] = "0.6"
os.environ["LLM_MAX_TOKENS"] = "600"

# 4. Turn-Taking & VAD (Silence cutoff for fast conversational flow)
os.environ["ASR_VAD_MODE"] = "server"             # Silero neural VAD (blocks fan/keyboard noise)
os.environ["VOICE_SILERO_SILENCE_MS"] = "550"     # 550ms silence = user finished speaking (was 1200ms!)
os.environ["VOICE_AUTO_SEND_MS"] = "550"
os.environ["VOICE_SPEAKER_GATE"] = "0"            # Disabled so students/users aren't rejected

# 5. ASR (faster-whisper on CUDA fp16)
os.environ["ASR_BACKEND"] = "faster-whisper"
os.environ["ASR_MODEL"] = "large-v3-turbo"        # Best Hindi/Hinglish accuracy (only ~1.6 GB VRAM)
os.environ["ASR_LANG"] = "hi"
os.environ["ASR_DEVICE"] = "cuda"
os.environ["ASR_COMPUTE"] = "float16"
os.environ["ASR_FINAL_BEAM"] = "1"                # Greedy is 4x faster (~80ms) and large-v3-turbo doesn't need beam 5
os.environ["ASR_INITIAL_PROMPT"] = "नमस्ते राहुल भाई, आप कैसे हैं? हाँ, मैं पूछ रहा हूँ कि मेरी आवाज़ आ रही है?"
# 6. TTS (OmniVoice on CUDA fp16)
os.environ["VOICE_API_DEVICE"] = "cuda"
os.environ["VOICE_API_DTYPE"] = "fp16"
os.environ["VOICE_NUM_STEP"] = "16"               # Subsequent windows get high quality (runs in background)
os.environ["VOICE_FIRST_STEP"] = "7"              # Sweet spot: 7 steps on T4 takes ~160ms and sounds clean
os.environ["VOICE_FIRST_WINDOW_CHARS"] = "26"     # ~4-6 words: enough text for natural intonation
os.environ["VOICE_MIN_WINDOW_CHARS"] = "24"
os.environ["VOICE_WINDOW_CHARS"] = "85"
os.environ["VOICE_TEMPERATURE"] = "0.3"
os.environ["VOICE_SPEED"] = "1.02"                # 2% faster gives a snappier conversational cadence

# 7. Voice Clone Reference
os.environ["VOICE_REF_TEXT"] = "कोडिंग में बहुत मज़ा आता है, बट समटाइम्स बग्स आर सो अनोइंग यार।"
os.environ["VOICE_REF_AUDIO"] = "/kaggle/input/datasets/rajsingha/bunty-updated/my_voice.wav"

# 8. Screen understanding (VLM context engine) — LOCAL Qwen2.5-VL-3B in-process
os.environ["VISION_BACKEND"] = "local"             # force the 3B model (no API key, no cost)

print("🔥 Realtime Conversational Voice Pipeline Configured for Kaggle GPU!")

🔥 Realtime Conversational Voice Pipeline Configured for Kaggle GPU!


In [68]:
!node --version && cd web/ui && npm install --silent && npm run build

v20.19.0

> voice-ui@0.0.0 build
> vite build

vite v6.4.3 building for production...
transforming (1) src/main.jsxtransforming (7) src/audioEngine.jstransforming (20) node_modules/react-icons/lib/index.mjstransforming (27) node_modules/react-dom/index.js✓ 34 modules transformed.
rendering chunks (1)...computing gzip size (0)...computing gzip size (1)...computing gzip size (2)...computing gzip size (3)...dist/index.html                   0.44 kB │ gzip:  0.32 kB
dist/assets/index-DxFv4rk6.css   25.33 kB │ gzip:  5.59 kB
dist/assets/index-BFM9DDMd.js   178.90 kB │ gzip: 58.76 kB
✓ built in 1.53s
⠙

In [69]:
!pip -q install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("1b4EWhlNt3eQvoNL09GXdkkLUGS_5zPS1AU5cXQgY6xVu53D6")
print("PUBLIC URL:", ngrok.connect(8000))


PUBLIC URL: NgrokTunnel: "https://68ff-34-80-244-103.ngrok-free.app" -> "http://localhost:8000"


In [ ]:
!python voice_api.py


INFO:     Started server process [2213]
INFO:     Waiting for application startup.
2026-09-06 21:34:16,828 Device: cuda | dtype: torch.float16
Fetching 13 files: 100%|█████████████████████| 13/13 [00:00<00:00, 13383.89it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100%|██████████████████████| 527/527 [00:00<00:00, 1463.75it/s]
2026-09-06 21:34:20,627 OmniVoice loaded + voice prompt cached from my_voice.wav
2026-09-06 21:34:20,628 Loading ASR backend 'faster-whisper' (model 'large-v3-turbo')...
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
2026-09-06 21:34:20,842 HTTP Request: GET https://huggingface.co/api/models/mobiuslabsgmbh/faster-whisper-large-v3-turbo/revision/main "HTTP/1.1 307 Temporary Redirect"
2026-09-06 21:34:21,035 HTTP Request: GET https://huggingface.co/api/models/dropbox-dash/faster-whisper-large-v3-turbo/revision/main "HTTP/1.1 200 OK"
2026-09-06 21:34:22,486 ASR backend 'f